# Generator-level dijet frame and forward/backward check

Read one Embedding, standard PYTHIA pPb, or PYTHIA CP5 production and plot generator-level dijet pseudorapidity in the CM and Lab frames for either the standard or compact test $p_T^{ave}$ bins. A separate canvas shows the CM-frame Forward/Backward ratio in every interval.

### Detailed workflow and physics conventions

For each half-open interval $[p_{T,low}^{ave},p_{T,high}^{ave})$, `hGenDijetPtEtaCM_{index}` and `hGenDijetPtEtaLab_{index}` are projected onto pseudorapidity. Each frame is normalized independently according to `NORMALIZATION`; the default unit bin-content sum compares shapes and removes absolute weighted-yield information. In pPb files, Lab is the stored beam-oriented Lab definition and CM includes the pPb boost. In pp CP5 files, Lab and CM are both stored detector eta and should agree within numerical precision.

The CM observable is $F/B(|\eta_{CM}|)=N(+|\eta_{CM}|)/N(-|\eta_{CM}|)$. It is built from unnormalized Forward and Backward yields folded from the full CM projection. These distinct weighted populations use ROOT standard independent-error propagation; binomial division is neither used nor configurable. Empty denominator bins retain ROOT's standard behavior.

## Environment and imports

Start Jupyter from the repository root with `py-env/bin/python -m jupyter notebook`.

In [1]:
# Cell role: initialize the reproducible Python/ROOT environment and shared helpers.
# Interpretation: No physics histogram is modified here; ROOT ownership is configured before files open.
# The preceding Markdown gives the equations and physics assumptions for this step.
%load_ext autoreload
%autoreload 2

from pathlib import Path
import os
import sys
from IPython.display import display

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'CMakeLists.txt').is_file() and (p / 'hist_analysis').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Cannot locate jetAnalysis. Start Jupyter from its root.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hist_analysis.python.notebook_setup import load_root
ROOT = load_root(batch=True)
from hist_analysis.config.files import BASE_DIR
from hist_analysis.config.histograms import DIJET_DELTA_PHI_SELECTION_LABEL, DIJET_PTAVE_BINS, STANDARD_DIJET_ETA_CUT_INDEX, TEST_DIJET_PTAVE_BINS
from hist_analysis.python.dijet_closures import DijetClosureCurve, build_dijet_gen_comparisons
from hist_analysis.python.histogram_io import load_histogram, resolve_combined_file, resolve_direction_file
from hist_analysis.python.histogram_ops import normalize_histogram
from hist_analysis.python.plotting import draw_closure, draw_overlay
from hist_analysis.python.plot_labels import CMS_SIMULATION_HEADER, PPB_HEADER_RIGHT
from hist_analysis.python.projections import project_semantic_th2
from hist_analysis.python.histogram_production import resolve_catalog_file
from hist_analysis.python.root_style import configure_cms_header

CMS_HEADER_LEFT = CMS_SIMULATION_HEADER
CMS_HEADER_RIGHT = PPB_HEADER_RIGHT
configure_cms_header(CMS_HEADER_LEFT, CMS_HEADER_RIGHT)

ROOT.gStyle.SetOptStat(0)
ROOT.gStyle.SetPalette(ROOT.kBird)
ROOT.TH1.AddDirectory(False)

from hist_analysis.python.root_style import configure_mplhep_export

SAVE_MPLHEP = False  # retain ROOT output and optionally add mplhep PDFs
SAVE_MPLHEP_PNG = False
configure_mplhep_export(SAVE_MPLHEP, save_png=SAVE_MPLHEP_PNG)


## Configuration

Choose `SAMPLE='embedding'`, `'pythia'`, or `'cp5'`. `DIRECTION` selects a standard pPb direction (or combined file) and is ignored for CP5. `PTAVE_BIN_SET` switches between canonical analysis bins and compact test bins. Index 5 is the standard stored $|\eta_{CM}^{jet}|<1.9$ selection. `CP5_FILE` and the output directory can be overridden with environment variables.

In [2]:
# Cell role: define and validate user-facing analysis configuration and resolve the input.
# Interpretation: These values select the production, physics acceptance, pTave binning, normalization, and outputs.
# The preceding Markdown gives the equations and physics assumptions for this step.
# SAMPLE = 'embedding'       # embedding, pythia, or cp5
SAMPLE = 'cp5'       # embedding, pythia, or cp5
DIRECTION = 'combined'      # Pbgoing, pgoing, or combined; ignored for cp5
FILE_STEM = 'jetId'
CP5_FILE = Path(os.environ.get('CP5_FILE', resolve_catalog_file('official_mc__pythia_cp5__pp')))
from hist_analysis.config.histograms import DIJET_ETA_CUTS, STANDARD_DIJET_ETA_CUT_INDEX
ETA_CUTS = DIJET_ETA_CUTS  # shared current production cut order
ETA_CUT_INDEX = STANDARD_DIJET_ETA_CUT_INDEX
PTAVE_BIN_SET = 'test'     # standard or test
REBIN_ETA = 2
NORMALIZATION = 'integral' # none, integral, or bin_width
ETA_Y_RANGE = None         # None selects one automatic range for all frame plots
FRAME_RATIO_RANGE = (0.5, 1.5)
FB_RANGE = (0.75, 1.25)
DRAW_GRID = True
SAVE_PNG = False
OUTPUT_DIR = Path(os.environ.get('MC_FB_GENERATOR_CHECK_OUTPUT_DIR', PROJECT_ROOT / 'hist_analysis' / 'output' / 'mc_fb_generator_check'))

if SAMPLE not in {'embedding', 'pythia', 'cp5'}:
    raise ValueError("SAMPLE must be 'embedding', 'pythia', or 'cp5'")
if DIRECTION not in {'Pbgoing', 'pgoing', 'combined'}:
    raise ValueError("DIRECTION must be 'Pbgoing', 'pgoing', or 'combined'")
if PTAVE_BIN_SET not in {'standard', 'test'}:
    raise ValueError("PTAVE_BIN_SET must be 'standard' or 'test'")
if ETA_CUT_INDEX not in range(len(ETA_CUTS)):
    raise ValueError(f'ETA_CUT_INDEX must be in [0, {len(ETA_CUTS)-1}]')
if isinstance(REBIN_ETA, bool) or not isinstance(REBIN_ETA, int) or REBIN_ETA < 1:
    raise ValueError('REBIN_ETA must be a positive integer')
if NORMALIZATION not in {'none', 'integral', 'bin_width'}:
    raise ValueError("NORMALIZATION must be 'none', 'integral', or 'bin_width'")
PTAVE_BINS = tuple(DIJET_PTAVE_BINS if PTAVE_BIN_SET == 'standard' else TEST_DIJET_PTAVE_BINS)
ETA_CUT = ETA_CUTS[ETA_CUT_INDEX]

if SAMPLE == 'cp5':
    INPUT_FILE, SAMPLE_LABEL, DIRECTION_LABEL = CP5_FILE, 'PYTHIA CP5 (pp)', 'pp'
elif DIRECTION == 'combined':
    INPUT_FILE = resolve_combined_file(BASE_DIR, SAMPLE, FILE_STEM)
    SAMPLE_LABEL, DIRECTION_LABEL = ('Embedding' if SAMPLE == 'embedding' else 'PYTHIA'), 'Combined'
else:
    INPUT_FILE = resolve_direction_file(BASE_DIR, SAMPLE, DIRECTION, FILE_STEM)
    SAMPLE_LABEL = 'Embedding' if SAMPLE == 'embedding' else 'PYTHIA'
    DIRECTION_LABEL = 'Pb-going' if DIRECTION == 'Pbgoing' else 'p-going'
if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Missing configured ROOT file: {INPUT_FILE}')
print(f'{SAMPLE_LABEL}, {DIRECTION_LABEL}: {INPUT_FILE}')
print(f'pTave bins ({PTAVE_BIN_SET}): {PTAVE_BINS}')

PYTHIA CP5 (pp), pp: /Users/gnigmat/cernbox/ana/pPb8160/pythia/cp5tune/HiForestMiniAOD_hist.root
pTave bins (test): ((50, 100), (100, 180), (180, 300), (300, 500), (500, 1000))


## Projection and F/B construction

The shared dijet helper projects the full CM TH2 over half-open pTave intervals and folds its unnormalized eta bins into Forward and Backward, checks matching bin edges, and constructs F/B with `TH1::Divide(..., '')`. The Lab projection uses the same semantic TH2 projection and normalization helpers. All histograms are detached from their input file.

In [3]:
# Cell role: project both frames and construct CM F/B for every configured pTave interval.
# Interpretation: Frame shapes may be normalized; F/B always divides unnormalized weighted yields.
# The preceding Markdown gives the equations and physics assumptions for this step.
GEN_CURVE = DijetClosureCurve('Gen', 'hGenDijetPtEtaCM_{eta_cut_index}')
LAB_KEY = f'hGenDijetPtEtaLab_{ETA_CUT_INDEX}'
PROJECTION_CACHE = {}
for interval_index, ptave_range in enumerate(PTAVE_BINS):
    cm_shapes, fb_ratios, keys = build_dijet_gen_comparisons(
        INPUT_FILE, (GEN_CURVE,), eta_cut_index=ETA_CUT_INDEX, ptave_range=ptave_range,
        nominal='Gen', rebin_eta=REBIN_ETA, normalization=NORMALIZATION, ratio_option='',
    )
    lab_source = load_histogram(str(INPUT_FILE), LAB_KEY)
    if not lab_source.InheritsFrom('TH2'):
        raise TypeError(f'{LAB_KEY!r} in {INPUT_FILE} is not a TH2')
    lab_source.Rebin2D(1, REBIN_ETA)
    lab_projection = project_semantic_th2(lab_source, 'eta', ptave_range, name=f'gen_dijet_lab_{interval_index}')
    PROJECTION_CACHE[tuple(ptave_range)] = {
        'frames': {'CM': cm_shapes['Gen'], 'Lab': normalize_histogram(lab_projection, NORMALIZATION)},
        'forward_backward': fb_ratios['Gen'],
        'keys': {'cm': keys['Gen']['cm'], 'lab': LAB_KEY, 'forward': keys['Gen']['forward'], 'backward': keys['Gen']['backward']},
    }

if ETA_Y_RANGE is None:
    eta_upper = max(h.GetBinContent(i) + h.GetBinError(i) for entry in PROJECTION_CACHE.values() for h in entry['frames'].values() for i in range(1, h.GetNbinsX()+1))
    COMMON_ETA_Y_RANGE = (0.0, 1.25 * eta_upper)
else:
    COMMON_ETA_Y_RANGE = ETA_Y_RANGE
print('Common frame-distribution y range:', COMMON_ETA_Y_RANGE)

Common frame-distribution y range: (0.0, 0.15903934143780585)


## CM/Lab distributions and CM Forward/Backward ratios

Each interval produces one independently normalized frame overlay with Lab/CM in a lower ratio panel, plus one unnormalized-yield F/B plot. The frame ratio uses standard independent-error propagation and defaults to the range 0.5--1.5. For CP5, the CM/Lab comparison is specifically a no-boost consistency check.

In [4]:
# Cell role: draw and save the frame overlays and CM F/B plots.
# Interpretation: Frame differences test coordinate conventions; F/B unity indicates equal forward and reflected-backward yields.
# The preceding Markdown gives the equations and physics assumptions for this step.
frame_results, fb_results = {}, {}
y_titles = {'none': 'Weighted yield', 'integral': '1/N dN/d#eta^{dijet}', 'bin_width': '1/N dN/d#eta^{dijet}'}
sample_tag = f'{SAMPLE}_{DIRECTION if SAMPLE != "cp5" else "pp"}'
for low, high in PTAVE_BINS:
    ptave_range, entry = (low, high), PROJECTION_CACHE[(low, high)]
    ptave_tag = f'{low:g}_{high:g}'.replace('.', 'p')
    annotations = (
        'pPb 8.16 TeV' if SAMPLE != 'cp5' else 'pp 8.16 TeV', SAMPLE_LABEL, DIRECTION_LABEL, 'Gen dijets',
        f'{low:g} < p_{{T}}^{{ave}} < {high:g} GeV', f'|#eta_{{CM}}^{{jet}}| < {ETA_CUT:g}',
        'p_{T}^{Lead} > 50 GeV', 'p_{T}^{SubLead} > 40 GeV', DIJET_DELTA_PHI_SELECTION_LABEL,
    )
    frame_tag = f'{sample_tag}_gen_frames_ptave_{ptave_tag}'
    frame_canvas, frame_ratios = draw_closure(
        entry['frames'], 'CM', title='', x_title='#eta^{dijet}', y_title=y_titles[NORMALIZATION],
        ratio_range=FRAME_RATIO_RANGE, ratio_option='', draw_nominal_ratio=False,
        x_range=(-ETA_CUT-0.1, ETA_CUT+0.1), y_range=COMMON_ETA_Y_RANGE, annotations=annotations,
        grid=DRAW_GRID, style_indices={'CM': 0, 'Lab': 1}, output=OUTPUT_DIR / f'{frame_tag}.pdf',
        save_png=SAVE_PNG, canvas_name=frame_tag,
    )
    fb_tag = f'{sample_tag}_gen_fb_ptave_{ptave_tag}'
    fb_canvas = draw_overlay(
        {'Gen F/B': entry['forward_backward']}, title='', x_title='#eta_{CM}^{dijet}',
        y_title='Forward / Backward', x_range=(0.0, ETA_CUT+0.1), y_range=FB_RANGE, annotations=annotations,
        grid=DRAW_GRID, reference_y=1.0, style_indices={'Gen F/B': 2}, output=OUTPUT_DIR / f'{fb_tag}.pdf',
        save_png=SAVE_PNG, canvas_name=fb_tag,
    )
    frame_results[ptave_range] = {'canvas': frame_canvas, 'lab_over_cm': frame_ratios['Lab'], **entry}
    fb_results[ptave_range] = {'canvas': fb_canvas, 'forward_backward': entry['forward_backward'], 'keys': entry['keys']}
    print(ptave_range, entry['keys'])
    display(frame_canvas)
    display(fb_canvas)

(50, 100) {'cm': 'hGenDijetPtEtaCM_5', 'lab': 'hGenDijetPtEtaLab_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_frames_ptave_50_100.pdf has been created


(100, 180) {'cm': 'hGenDijetPtEtaCM_5', 'lab': 'hGenDijetPtEtaLab_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}


(180, 300) {'cm': 'hGenDijetPtEtaCM_5', 'lab': 'hGenDijetPtEtaLab_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_fb_ptave_50_100.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_frames_ptave_100_180.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_fb_ptave_100_180.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_frames_ptave_180_300.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_fb_ptave_180_300.pdf has been created


(300, 500) {'cm': 'hGenDijetPtEtaCM_5', 'lab': 'hGenDijetPtEtaLab_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}


(500, 1000) {'cm': 'hGenDijetPtEtaCM_5', 'lab': 'hGenDijetPtEtaLab_5', 'forward': 'hGenDijetPtEtaForward_5', 'backward': 'hGenDijetPtEtaBackward_5'}


Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_frames_ptave_300_500.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_fb_ptave_300_500.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_frames_ptave_500_1000.pdf has been created
Info in <TCanvas::Print>: pdf file /Users/gnigmat/work/cms/jetAnalysis/hist_analysis/output/mc_fb_generator_check/cp5_pp_gen_fb_ptave_500_1000.pdf has been created
